# Note:
This version of the tutorial was written and tested for `pyaerocom==0.30.1` / `pya-v2025.05.13.conda`.

# Setup
To start evaluating with Pyaerocom, we first need a functioning installation of Pyaerocom. There are two options here, either using a prepared module or by installing with pip. For this tutorial we will be using the module, though I'll briefly mention how to install via pip as well.

## Module
The pyaerocom modules can be found in the folder `/modules/MET/rhel8/user-modules/fou-kl/aerotools` and can be activated as follows:

In [1]:
%%bash
module use /modules/MET/rhel8/user-modules

ls /modules/MET/rhel8/user-modules/fou-kl/aerotools # Listing available modules

module load /modules/MET/rhel8/user-modules/fou-kl/aerotools/pya-v2025.05.13.conda # Loading a module. Here we use the most recent as of writing.

# Check that modules was loaded successfully:
pya --version

bash: line 1: module: command not found
ls: cannot access '/modules/MET/rhel8/user-modules/fou-kl/aerotools': No such file or directory
bash: line 5: module: command not found


CalledProcessError: Command 'b'module use /modules/MET/rhel8/user-modules\n\nls /modules/MET/rhel8/user-modules/fou-kl/aerotools # Listing available modules\n\nmodule load /modules/MET/rhel8/user-modules/fou-kl/aerotools/pya-v2025.05.13.conda\n'' returned non-zero exit status 127.

## Via pip
The below shows how to setup a virtual environment and installing pyaerocom from various places. Usually this should not be necessary to use Pyaerocom. This may be useful to run for example development version in between releases in case of a critical bug fix.

In [ ]:
%%bash
python -m venv venv # Will create a virtual environment in the folder 'venv'.
source venb/bin/activate # Activate the virtual environment

# From pypi
pip install "pyaerocom==0.30.1"
pip install "pyaerocom" # Omit version number for latest release.

# From Github
pip install git+https://github.com/metno/pyaerocom.git
pip install git+https://github.com/metno/pyaerocom.git@tag # from a tag, see https://github.com/metno/pyaerocom/tags
pip install git+https://github.com/metno/pyaerocom.git@branch # from a branch.

pya --version # Verify successfull installation
#>🦄 pyaerocom 0.30.1

# deactivate # To deactivate the virtual environment.

# Running an Evaluation
A simple configuration to run, which can help verify that pyaerocom is working as expected is the config benchmark which can be found [here](https://gitlab.met.no/aeroval/config/-/blob/master/config_files/testing/cfg_benchmark.py):

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Config file for Pyaeroval benchmarking
"""
import os
### Define filters for the obs subsets
BASE_FILTER = {
    "latitude": [-90, 90],
    "longitude": [-180, 180],
    "station_id": ["NO0042*"],
    "negate": "station_id",
}
preface = ""
MODELS = {
    "ECMWF-CAMS-REAN": dict(
        model_id="ECMWF_CAMS_REAN",
    )
}
OBS_GROUNDBASED = {
    ##################
    #    AERONET
    ##################
    "AERONET": dict(
        obs_id="AeronetSunV3Lev1.5.daily",
        obs_vars=[
            "od550aer",
        ],
        web_interface_name="AERONET",
        obs_vert_type="Column",
        ignore_station_names="DRAGON*",
        ts_type="daily",
        colocate_time=True,
        min_num_obs=dict(
            yearly=dict(
                monthly=9,
                daily=90,
            ),
            monthly=dict(
                weekly=1,
            ),
            weekly=dict(
                daily=3,
            ),
        ),
        obs_filters=BASE_FILTER,
    ),
    ##################
    #    EBAS
    ##################
    "EBAS": dict(
        obs_id="EBASMC",
        web_interface_name="EBAS",
        obs_vars=["concpm10"],
        obs_vert_type="Surface",
        ts_type="daily",
        colocate_time=True,
        obs_filters=BASE_FILTER,
    ),
}
# Setup for supported satellite evaluations
OBS_SAT = {}
OBS_CFG = {**OBS_GROUNDBASED, **OBS_SAT}
DEFAULT_RESAMPLE_CONSTRAINTS = dict(
    yearly=dict(monthly=9),
    monthly=dict(
        daily=21,
        weekly=3,
    ),
    daily=dict(hourly=1),
)
CFG = dict(
    model_cfg=MODELS,
    obs_cfg=OBS_CFG,
    obs_only=False,
    json_basedir=os.path.abspath("./data"),
    coldata_basedir=os.path.abspath("./coldata"),
    # if True, existing colocated data files will be deleted
    reanalyse_existing=True,
    only_json=False,
    add_model_maps=False,
    only_model_maps=False,
    clear_existing_json=True,
    # if True, the analysis will stop whenever an error occurs (else, errors that
    # occurred will be written into the logfiles)
    raise_exceptions=False,
    # Regional filter for analysis
    filter_name="ALL-wMOUNTAINS",
    # colocation frequency (no statistics in higher resolution can be computed)
    ts_type="monthly",
    map_zoom="World",
    freqs=["daily", "monthly", "yearly"],
    periods=[
        "2013",
    ],
    main_freq="monthly",
    zeros_to_nan=False,
    min_num_obs=DEFAULT_RESAMPLE_CONSTRAINTS,
    colocate_time=True,
    resample_how={"vmro3": {"daily": {"hourly": "max"}}},
    obs_remove_outliers=False,
    model_remove_outliers=False,
    harmonise_units=True,
    regions_how="htap",
    annual_stats_constrained=False,
    proj_id="testing",
    exp_id="benchmark",
    exp_name="benchmark",
    exp_descr=(
        "Benchmark a typical PyAeroval experiment for PyAerocom development testing"
    ),
    exp_pi="Pyaerocom Developers",
    public=True,
    weighted_stats=True,
    obs_only_stats=False,
    var_order_menu=["od550aer", "concpm10"],
)
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    import pyaerocom as pya
    from pyaerocom import const
    from pyaerocom.aeroval import EvalSetup, ExperimentProcessor
    plt.close("all")
    stp = EvalSetup(**CFG)
    ana = ExperimentProcessor(stp)
    res = ana.run()

TypeError: 'coroutine' object does not support item assignment

# Reading Observation Data With Pyaro

# Parallelization

# Aeroval-test
[Aeroval-test](https://aeroval-test.met.no) is the website intended for displaying the results of an evaluation. To be able to access you data it needs to be made available from in `/lustre/storeB/project/fou/kl/emep/Web/AeroVal`. It is recommended to symlink a folder with your user name to a more convenient place to store this file.

The following commands show how to setup a directory for your evaluations. The actual files that are hosted will then be accessible to you at `/lustre/storeB/users/USERNAME/aeroval-test/data`.

Once completed evaluations will be available on [https://aeroval-test.met.no/USERNAME](https://aeroval-test.met.no/USERNAME).

In [ ]:
%%bash
cd /lustre/storeB/users/USERNAME

mkdir -p aeroval-test/data # This is the directory that will be used as json_basedir in the evaluation.

cd /lustre/storeB/project/fou/kl/emep/Web/AeroVal
ln -s /lustre/storeB/users/USERNAME/aeroval-test USERNAME